# Process 5° MODIS + S1 into training patches

**Colab runtime:** **Runtime → Change runtime type → GPU** (e.g. **T4** or **A100**) when you plan to **train** in the same session. This notebook step is mostly **CPU + disk I/O**; a GPU does **not** speed up reading GeoTIFFs from Drive. If you only run preprocessing, **CPU** is fine. Use **high-RAM** if you hit out-of-memory errors.

Reads MODIS GeoTIFFs and S1 `.npy` from your project **`data/` on Google Drive**, builds 10-step MODIS stacks + flood labels, slices **64×64** patches, and writes **`data/processed_tiles/`** on **Google Drive** (same project folder — that is your persistent output).

**Drive-only reads:** Set **`USE_LOCAL_COPY = False`** in the setup cell to read raw MODIS/S1 **only** from Drive (no temporary copy under `/content`). **Tradeoff:** GeoTIFF access over the Drive mount is **much slower** (often many minutes per sample). **`USE_LOCAL_COPY = True`** only copies inputs to Colab’s **temporary** disk to read faster; **processed tiles still save to Drive**, not to your PC.

Shards are saved **uncompressed** (`.npz`) for memory-mapped training. Each shard stores **`KEY`** `(N, 3)` int32 per patch row — `(year, period_idx, tile_id)` — so **`ShardedPatchDataModule`** can do **leave-one-year-out** splits without re-reading GeoTIFFs. The loop **writes shards often** (default: every **1000** patches and every **5** successful tiles), updates **`manifest.json` after each shard**, and **flushes remaining data in `finally`** if you stop the cell — so partial runs still leave files on Drive.


**Resume:** Completed tiles are listed in **`data/processed_tiles/processed_tile_keys.json`**. Re-running skips those tiles and appends new **`patch_shard_*.npz`** (using **`manifest.json`** for the next shard index). To redo everything, delete **`processed_tile_keys.json`**, **`manifest.json`**, and **`patch_shard_*.npz`** in that folder.

**Drive disconnect (Errno 107):** If you see *Transport endpoint is not connected*, the Colab **Google Drive mount dropped**. Remount with `drive.mount('/content/drive', force_remount=True)`, `%cd` to your project, then continue.

**RAM:** MODIS/S1 use a **bounded LRU cache**; **`gc.collect()`** runs after heavy steps so memory is freed. If you still OOM, lower **`MODIS_CACHE_MAX`** / **`S1_CACHE_MAX`** in the `load_sample` cell.
Then run **`train_from_5d_data.ipynb`**.


In [1]:
# Colab: mount Drive, install deps
from google.colab import drive
drive.mount("/content/drive")

!pip install -q rioxarray scipy tqdm pyyaml pysheds
%cd /content/drive/MyDrive/indonesia-flood-cnn-lstm


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/indonesia-flood-cnn-lstm


In [2]:
import sys
import json
import shutil
from pathlib import Path
import numpy as np

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# False = read raw MODIS/S1 only from Google Drive (slow). True = faster reads from Colab temp disk; outputs still go to Drive.
USE_LOCAL_COPY = False
RAW_MODIS_SUBDIR = "modis_5d"
RAW_S1_SUBDIR = "s1_labels_5d"

DATA_SRC = PROJECT_ROOT / "data"
OUT_DIR = DATA_SRC / "processed_tiles"
OUT_DIR.mkdir(parents=True, exist_ok=True)

READ_FROM = "drive"
if USE_LOCAL_COPY:
    import time
    local_root = Path("/content/data_raw_local")
    local_root.mkdir(parents=True, exist_ok=True)
    try:
        for sub in (RAW_MODIS_SUBDIR, RAW_S1_SUBDIR):
            src = DATA_SRC / sub
            dst = local_root / sub
            if src.exists() and not dst.exists():
                t0 = time.perf_counter()
                print("Copying", sub, "to /content (one-time, may take several minutes)...")
                shutil.copytree(src, dst)
                print(f"  done in {time.perf_counter() - t0:.0f}s")
        DATA_DIR = local_root
        READ_FROM = "local"
    except OSError as e:
        print("Local copy failed, reading from Drive only:", e)
        DATA_DIR = DATA_SRC
else:
    DATA_DIR = DATA_SRC
    print("WARNING: USE_LOCAL_COPY=False — expect very slow GeoTIFF reads from Drive.")

MODIS_DIR = DATA_DIR / RAW_MODIS_SUBDIR
S1_DIR = DATA_DIR / RAW_S1_SUBDIR

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Read MODIS/S1 from:", DATA_DIR, f"({READ_FROM} SSD)" if READ_FROM == "local" else "(Drive — slow)")
print("MODIS_DIR:", MODIS_DIR)
print("S1_DIR:", S1_DIR)
print("OUT_DIR (Drive):", OUT_DIR)


PROJECT_ROOT: /content/drive/MyDrive/indonesia-flood-cnn-lstm
Read MODIS/S1 from: /content/drive/MyDrive/indonesia-flood-cnn-lstm/data (Drive — slow)
MODIS_DIR: /content/drive/MyDrive/indonesia-flood-cnn-lstm/data/modis_5d
S1_DIR: /content/drive/MyDrive/indonesia-flood-cnn-lstm/data/s1_labels_5d
OUT_DIR (Drive): /content/drive/MyDrive/indonesia-flood-cnn-lstm/data/processed_tiles


In [3]:
def parse_modis_name(fname):
    stem = Path(fname).stem
    if not stem.startswith("Indo_5d_MODIS_"):
        return None
    parts = stem.replace("Indo_5d_MODIS_", "").split("_")
    if len(parts) < 4:
        return None
    try:
        return int(parts[0]), int(parts[1]), int(parts[-1])
    except ValueError:
        return None

def parse_s1_name(fname):
    stem = Path(fname).stem
    if not stem.startswith("Indo_5d_S1_"):
        return None
    parts = stem.replace("Indo_5d_S1_", "").split("_")
    if len(parts) < 4:
        return None
    try:
        return int(parts[0]), int(parts[1]), int(parts[-1])
    except ValueError:
        return None

modis_files = {parse_modis_name(f.name): f for f in MODIS_DIR.glob("*.tif") if parse_modis_name(f.name)}
s1_files = {parse_s1_name(f.name): f for f in S1_DIR.glob("*.npy") if parse_s1_name(f.name)}
common_keys = set(modis_files.keys()) & set(s1_files.keys())
print(f"MODIS: {len(modis_files)}  S1: {len(s1_files)}  common: {len(common_keys)}")


MODIS: 5290  S1: 5283  common: 5283


In [4]:
from collections import OrderedDict
from datetime import datetime, timedelta
import gc
import rioxarray
from scipy.ndimage import zoom
import rasterio

def _count_8day_periods(year):
    d = datetime(year, 1, 1)
    n = 0
    while d.year == year:
        n += 1
        d += timedelta(days=8)
    return n

_PERIOD_COUNT = {}

def get_8day_periods(year):
    if year not in _PERIOD_COUNT:
        _PERIOD_COUNT[year] = _count_8day_periods(year)
    return _PERIOD_COUNT[year]

SEQUENCE_LENGTH = 10
PATCH_SIZE = 64
PATCH_STRIDE = 64
S1_PERCENTILE = 20

# Bounded LRU cache — unbounded dicts keep every GeoTIFF in RAM forever
MODIS_CACHE_MAX = 20
S1_CACHE_MAX = 40
_modis_cache = OrderedDict()
_s1_cache = OrderedDict()


def _trim(cache, max_size):
    while len(cache) > max_size:
        cache.popitem(last=False)


def load_sample(year, p_idx, tile_id):
    modis_list = []
    for t in range(SEQUENCE_LENGTH):
        pp = p_idx - (SEQUENCE_LENGTH - 1 - t)
        yy = year
        while pp < 0:
            yy -= 1
            pp += get_8day_periods(yy)
        while pp >= get_8day_periods(yy):
            pp -= get_8day_periods(yy)
            yy += 1
        key = (yy, pp, tile_id)
        if key not in modis_files:
            return None
        if key in _modis_cache:
            _modis_cache.move_to_end(key)
            arr = _modis_cache[key]
        else:
            try:
                da = rioxarray.open_rasterio(modis_files[key])
                arr = np.asarray(da).squeeze().astype(np.float32, copy=False)
            except (rasterio.errors.RasterioIOError, OSError):
                return None
            _modis_cache[key] = arr
            _modis_cache.move_to_end(key)
            _trim(_modis_cache, MODIS_CACHE_MAX)
        modis_list.append(arr)
    key = (year, p_idx, tile_id)
    if key not in s1_files:
        return None
    if key in _s1_cache:
        _s1_cache.move_to_end(key)
        s1 = _s1_cache[key]
    else:
        try:
            s1 = np.load(s1_files[key], allow_pickle=False).astype(np.float32, copy=False)
        except Exception as e:
            # Corrupt/truncated .npy or any load failure — skip tile, keep processing
            print(f"Skipping bad S1 .npy: {s1_files[key]} — {e}")
            return None
        _s1_cache[key] = s1
        _s1_cache.move_to_end(key)
        _trim(_s1_cache, S1_CACHE_MAX)
    modis_stack = np.stack(modis_list, axis=0)
    del modis_list
    if modis_stack.ndim == 4 and modis_stack.shape[1] == 7:
        pass
    elif modis_stack.ndim == 4 and modis_stack.shape[-1] == 7:
        modis_stack = np.transpose(modis_stack, (0, 3, 1, 2))
    else:
        return None
    target_h, target_w = modis_stack.shape[2], modis_stack.shape[3]
    if s1.shape != (target_h, target_w):
        sy, sx = target_h / s1.shape[0], target_w / s1.shape[1]
        s1 = zoom(s1, (sy, sx), order=1)
    valid = np.isfinite(s1) & (s1 > -999)
    if valid.sum() < 100:
        return None
    thresh = np.percentile(s1[valid], S1_PERCENTILE)
    flood = np.where(valid, (s1 < thresh).astype(np.float32), np.nan)
    flood = np.nan_to_num(flood, nan=0.0)
    hand = np.zeros((target_h, target_w), dtype=np.float32)
    hand_expanded = np.repeat(hand[np.newaxis, np.newaxis, :, :], SEQUENCE_LENGTH, axis=0)
    modis_norm = modis_stack / 10000.0
    modis_norm = np.clip(np.nan_to_num(modis_norm, nan=0), 0, 1)
    del modis_stack
    X = np.concatenate([modis_norm, hand_expanded], axis=1)
    del modis_norm, hand_expanded
    Y = flood
    gc.collect()
    return X, Y





In [5]:
import json
import gc

from pipeline.processor import create_patches
from tqdm import tqdm

MAX_SAMPLES = None  # e.g. 200 for smoke test; None = all
SHARD_PATCHES = 1000
FLUSH_EVERY_N_TILES = 5
# float16 shards ~halve Drive space vs float32; training loads as float32.
SAVE_FLOAT16 = True

PROCESSED_KEYS_PATH = OUT_DIR / "processed_tile_keys.json"


def key_str(k):
    y, p, t = k
    return f"{y}_{p:03d}_{t}"


def load_processed_keys():
    if PROCESSED_KEYS_PATH.exists():
        return set(json.loads(PROCESSED_KEYS_PATH.read_text()))
    return set()


def save_processed_keys(s):
    PROCESSED_KEYS_PATH.write_text(json.dumps(sorted(list(s)), indent=0))


processed_keys = load_processed_keys()

mpath = OUT_DIR / "manifest.json"
if mpath.exists():
    m0 = json.loads(mpath.read_text())
    shard_idx = int(m0.get("n_shards", 0))
    total_patches = int(m0.get("total_patches", 0))
else:
    shard_idx = 0
    total_patches = 0

to_process = [k for k in sorted(common_keys) if k[1] >= SEQUENCE_LENGTH - 1]
if MAX_SAMPLES:
    to_process = to_process[:MAX_SAMPLES]
    print("MAX_SAMPLES:", MAX_SAMPLES)

before = len(to_process)
to_process = [k for k in to_process if key_str(k) not in processed_keys]
print(f"Resume: skipped {before - len(to_process)} tile(s) already on disk; {len(to_process)} tile(s) left to process.")

buf_chunks = []
n_skip = 0
tiles_ok = 0


def pending_patches():
    return sum(x.shape[0] for _, x, _ in buf_chunks)


def save_shard_arrays(X_arr, Y_arr, K_arr):
    global shard_idx, total_patches
    path = OUT_DIR / f"patch_shard_{shard_idx:04d}.npz"
    if SAVE_FLOAT16:
        X_arr = np.asarray(X_arr, dtype=np.float16)
        Y_arr = np.asarray(Y_arr, dtype=np.float16)
    K_arr = np.asarray(K_arr, dtype=np.int32)
    np.savez(path, X=X_arr, Y=Y_arr, KEY=K_arr)
    print(f"Wrote {path.name}: {X_arr.shape[0]} patches")
    total_patches += X_arr.shape[0]
    shard_idx += 1


def write_manifest():
    m = {
        "n_shards": shard_idx,
        "total_patches": total_patches,
        "patch_size": PATCH_SIZE,
        "sequence_length": SEQUENCE_LENGTH,
        "channels": 8,
        "has_patch_keys": True,
        "skipped_samples": n_skip,
        "n_tile_keys_done": len(processed_keys),
        "shard_files": [f"patch_shard_{i:04d}.npz" for i in range(shard_idx)],
        "partial": True,
    }
    (OUT_DIR / "manifest.json").write_text(json.dumps(m, indent=2))


def drain_n_patches(n):
    global buf_chunks
    parts_x, parts_y, parts_k = [], [], []
    keys_fully_done = []
    while n > 0 and buf_chunks:
        key, xp, yp = buf_chunks[0]
        c = xp.shape[0]
        yr, pi, tid = int(key[0]), int(key[1]), int(key[2])
        row = np.array([[yr, pi, tid]], dtype=np.int32)
        if c <= n:
            parts_x.append(xp)
            parts_y.append(yp)
            parts_k.append(np.repeat(row, c, axis=0))
            keys_fully_done.append(key)
            n -= c
            buf_chunks.pop(0)
        else:
            parts_x.append(xp[:n])
            parts_y.append(yp[:n])
            parts_k.append(np.repeat(row, n, axis=0))
            buf_chunks[0] = (key, xp[n:], yp[n:])
            n = 0
    if not parts_x:
        return
    X = np.concatenate(parts_x, axis=0)
    Y = np.concatenate(parts_y, axis=0)
    K = np.concatenate(parts_k, axis=0)
    del parts_x, parts_y, parts_k
    save_shard_arrays(X, Y, K)
    del X, Y
    for k in keys_fully_done:
        processed_keys.add(key_str(k))
    save_processed_keys(processed_keys)
    write_manifest()
    gc.collect()


try:
    for key in tqdm(to_process, desc="Process tiles"):
        try:
            year, p_idx, tile_id = key
            result = load_sample(year, p_idx, tile_id)
            if result is None:
                n_skip += 1
                continue
            X, Y = result
            x_patches = create_patches(X, patch_size=PATCH_SIZE, stride=PATCH_STRIDE)
            y_patches = create_patches(Y, patch_size=PATCH_SIZE, stride=PATCH_STRIDE)
            del X, Y
            if len(x_patches) == 0:
                continue
            buf_chunks.append((key, x_patches, y_patches))
            tiles_ok += 1

            while pending_patches() >= SHARD_PATCHES:
                drain_n_patches(SHARD_PATCHES)

            if tiles_ok % FLUSH_EVERY_N_TILES == 0 and pending_patches() > 0:
                drain_n_patches(pending_patches())
        except Exception as e:
            print(f"Skip tile {key}: {type(e).__name__}: {e}")
            n_skip += 1
            continue

finally:
    try:
        if pending_patches() > 0:
            drain_n_patches(pending_patches())
    except OSError as e:
        print("Drive I/O error while flushing buffer:", e)
        print("If you see 'Transport endpoint is not connected', remount Drive:")
        print("  from google.colab import drive")
        print("  drive.mount('/content/drive', force_remount=True)")
    mp = OUT_DIR / "manifest.json"
    try:
        if mp.exists():
            m = json.loads(mp.read_text())
            m["partial"] = False
            mp.write_text(json.dumps(m, indent=2))
    except OSError as e:
        print("Could not update manifest on Drive:", e)
        print("Errno 107 = Drive mount dropped. Remount with drive.mount(..., force_remount=True), then re-run.")
    print("Stopped or finished. manifest.json ->", OUT_DIR)
    print("total_patches:", total_patches, "shards:", shard_idx, "tile keys saved:", len(processed_keys), "skipped samples:", n_skip)





Resume: skipped 2829 tile(s) already on disk; 1426 tile(s) left to process.


Process tiles:   0%|          | 1/1426 [00:12<4:45:28, 12.02s/it]

Skipping bad S1 .npy: /content/drive/MyDrive/indonesia-flood-cnn-lstm/data/s1_labels_5d/Indo_5d_S1_2018_045_Tile_0.npy — cannot reshape array of size 249299936 into shape (50000,50000)


Process tiles:   0%|          | 5/1426 [00:40<3:02:52,  7.72s/it]

Wrote patch_shard_0961.npz: 1000 patches


...

Wrote patch_shard_1441.npz: 1000 patches


Process tiles: 100%|█████████▉| 1422/1426 [4:53:01<01:07, 16.81s/it]

Wrote patch_shard_1442.npz: 258 patches


Process tiles: 100%|██████████| 1426/1426 [4:53:27<00:00, 12.35s/it]


Wrote patch_shard_1443.npz: 171 patches
Stopped or finished. manifest.json -> /content/drive/MyDrive/indonesia-flood-cnn-lstm/data/processed_tiles
total_patches: 942974 shards: 1444 tile keys saved: 4253 skipped samples: 2
